# Benchmark v1 Trace Notebook

Notebook này chỉ tập trung vào **v1.0** và được thiết kế để trace từng bước xử lý:

- Xem template đầu vào.
- Sinh và nghe từng audio trung gian.
- Xem `input.wav` cuối cùng và annotation JSON.
- Chạy Gemini để tạo `output.wav` thật từ `input.wav`.
- Dùng ASR chuyển `output.wav` thành `output.json` để evaluator đọc được.
- Xem bảng đánh giá v1 theo từng sample.

Kịch bản mô phỏng chỉ tạo **đầu vào**. Đầu ra nên đến từ agent/model thật; mock output chỉ còn là fallback tắt/mở thủ công khi cần debug evaluator.


## 1. Clone repo và cài dependency tối thiểu

Chạy cell này nếu notebook đang ở Colab/Kaggle/máy mới. Nếu đã ở trong repo local thì cell vẫn tự tìm repo hiện tại.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import shutil

REPO_URL = "https://github.com/lamkdhe180931-arch/Full-Duplex-Bench.git"
BRANCH = "LamKD"
WORK_ROOT = Path(os.getenv("FDB_WORK_ROOT", "/content" if Path("/content").exists() else "/kaggle/working" if Path("/kaggle/working").exists() else Path.cwd())).resolve()
REPO_DIR = WORK_ROOT / "Full-Duplex-Bench"

if not (REPO_DIR / ".git").exists():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)

if shutil.which("ffmpeg") is None:
    print("Installing ffmpeg...")
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "pydub", "numpy", "edge-tts", "python-dotenv", "pandas",
    "google-genai", "soundfile", "scipy", "transformers", "accelerate", "torch"
], check=True)
print("Ready:", REPO_DIR)


## 1.1 API key setup

Cell này chuẩn bị `GEMINI_API_KEY` cho bước Gemini inference. Trên Kaggle nên dùng **Add-ons -> Secrets** rồi tạo secret tên `GEMINI_API_KEY`; notebook sẽ tự đọc qua `google_secret_manager` nếu có.


In [ ]:
import os

# Cách khuyên dùng trên Kaggle: Add-ons -> Secrets -> thêm secret GEMINI_API_KEY.
# Nếu không chạy trên Kaggle, có thể set biến môi trường trước khi chạy notebook.
GEMINI_API_KEY_MANUAL = ""  # Chỉ điền tạm nếu bạn test cá nhân; không commit/share notebook có key thật.
HF_TOKEN_MANUAL = ""  # Optional: giúp Hugging Face tải ASR ổn định hơn nếu gặp rate-limit.

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

if not os.getenv("GEMINI_API_KEY") and secrets is not None:
    try:
        secret_value = secrets.get_secret("GEMINI_API_KEY")
        if secret_value:
            os.environ["GEMINI_API_KEY"] = secret_value
    except Exception:
        pass

if not os.getenv("HF_TOKEN") and secrets is not None:
    try:
        secret_value = secrets.get_secret("HF_TOKEN")
        if secret_value:
            os.environ["HF_TOKEN"] = secret_value
    except Exception:
        pass

if not os.getenv("GEMINI_API_KEY") and GEMINI_API_KEY_MANUAL.strip():
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY_MANUAL.strip()
if not os.getenv("HF_TOKEN") and HF_TOKEN_MANUAL.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN_MANUAL.strip()

print("GEMINI_API_KEY:", "ready" if os.getenv("GEMINI_API_KEY") else "missing")
print("HF_TOKEN:", "ready" if os.getenv("HF_TOKEN") else "missing (optional)")


## 2. Setup trace v1

Mặc định notebook ghi ra thư mục preview riêng, không ghi đè dataset chính. Đặt `MAX_SAMPLES = None` để chạy hết template.

In [ ]:
from pathlib import Path
import contextlib
import io
import json
import math
import os
import sys
import shutil

import pandas as pd
from IPython.display import Audio, JSON, Markdown, display
from pydub import AudioSegment


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "v1_v1.5" / "data_generation" / "v1_0").exists():
            return base
    raise RuntimeError("Không tìm thấy repo root chứa v1_v1.5/data_generation/v1_0")

ROOT_DIR = find_repo_root()
DATA_GEN_DIR = ROOT_DIR / "v1_v1.5" / "data_generation"
if str(DATA_GEN_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_GEN_DIR))

from core.tts_generator import VietnameseTTSGenerator
from core.audio_mixer import AudioMixer

default_output = "/content/fdb_v1_trace" if Path("/content").exists() else "/kaggle/working/fdb_v1_trace" if Path("/kaggle/working").exists() else "/private/tmp/fdb_v1_trace"
OUTPUT_BASE = Path(os.getenv("FDB_V1_TRACE_OUTPUT", default_output)).resolve()
V1_OUTPUT = OUTPUT_BASE / "dataset" / "v1_0"
TEMPLATES_DIR = DATA_GEN_DIR / "v1_0" / "templates"
RESET_OUTPUT = True
MAX_SAMPLES = None  # None = chạy hết sample; đặt số nguyên nếu muốn test nhanh
TTS_SEED = int(os.getenv("FDB_TTS_SEED", "20260625"))

if RESET_OUTPUT and OUTPUT_BASE.exists():
    shutil.rmtree(OUTPUT_BASE)
V1_OUTPUT.mkdir(parents=True, exist_ok=True)

generator = VietnameseTTSGenerator(provider="edge-tts", seed=TTS_SEED)
mixer = AudioMixer()

print("ROOT_DIR:", ROOT_DIR)
print("V1_OUTPUT:", V1_OUTPUT)
print("MAX_SAMPLES:", "all" if MAX_SAMPLES is None else MAX_SAMPLES)


## 3. Helper hiển thị gọn

Các helper này là phần quan trọng để notebook dễ trace: bảng tóm tắt, audio player, JSON annotation, và waveform đơn giản.

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_template(name):
    items = read_json(TEMPLATES_DIR / f"{name}.json")
    return items if MAX_SAMPLES is None else items[:MAX_SAMPLES]


def quiet_generate(text, output_path, **kwargs):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with contextlib.redirect_stdout(io.StringIO()):
        generator.generate(text, str(output_path), **kwargs)
    return output_path


def save_audio(sound, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    mixer.save_audio(sound, str(output_path))
    return output_path


def audio_info(path):
    path = Path(path)
    sound = AudioSegment.from_file(path)
    return {
        "file": str(path),
        "duration_sec": round(len(sound) / 1000, 3),
        "frame_rate": sound.frame_rate,
        "channels": sound.channels,
        "dBFS": None if sound.dBFS == float("-inf") else round(sound.dBFS, 2),
    }


def show_audio(label, path):
    path = Path(path)
    display(Markdown(f"**{label}**"))
    display(pd.DataFrame([audio_info(path)]))
    display(Audio(filename=str(path)))


def show_json(label, path):
    display(Markdown(f"**{label}** `{Path(path).name}`"))
    display(JSON(read_json(path)))


def show_timeline(rows):
    display(pd.DataFrame(rows))


def sample_header(task, sample_id, description=""):
    display(Markdown(f"---\n### {task} / `{sample_id}`"))
    if description:
        display(Markdown(description))


def transcript_to_output_json(text, start, duration=1.4):
    words = text.split()
    if not words:
        return {"text": "", "chunks": []}
    step = max(duration / len(words), 0.15)
    chunks = []
    t = start
    for word in words:
        chunks.append({"text": word, "timestamp": [round(t, 3), round(t + step, 3)]})
        t += step
    return {"text": text, "chunks": chunks}


## 4. Xem template v1

Cell này chỉ hiển thị đầu vào kịch bản, không sinh audio.

In [ ]:
template_names = [
    "synthetic_pause_handling",
    "candor_turn_taking",
    "synthetic_user_interruption",
]
for name in template_names:
    items = load_template(name)
    display(Markdown(f"### `{name}` — {len(items)} samples"))
    display(pd.DataFrame(items))


## 5. Step trace: Pause Handling

Mỗi sample hiển thị:

1. `part_1.wav`
2. khoảng pause theo `pause_duration_sec` trong JSON
3. `part_2.wav`
4. `input.wav` cuối cùng
5. `pause.json` dùng cho đánh giá

In [ ]:
pause_rows = []
for item in load_template("synthetic_pause_handling"):
    sample_id = item["id"]
    if "pause_duration_sec" not in item:
        raise KeyError(f"{sample_id} missing required field: pause_duration_sec")
    pause_duration = float(item["pause_duration_sec"])
    sample_dir = V1_OUTPUT / "synthetic_pause_handling" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("synthetic_pause_handling", sample_id, item.get("description", ""))

    p1_path = quiet_generate(item["part_1"], sample_dir / "step_1_part_1.wav", role="primary")
    p1_profile = generator.last_synthesis.get("profile") if generator.last_synthesis else None
    p2_path = quiet_generate(item["part_2"], sample_dir / "step_3_part_2.wav", profile=p1_profile, role="primary")

    p1 = mixer.load_audio(str(p1_path))
    p2 = mixer.load_audio(str(p2_path))
    pause = AudioSegment.silent(duration=int(pause_duration * 1000), frame_rate=16000)
    input_sound = p1 + pause + p2
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    pause_info = [{"text": "[PAUSE]", "timestamp": [len(p1) / 1000.0, len(p1) / 1000.0 + pause_duration]}]
    pause_json_path = sample_dir / "pause.json"
    write_json(pause_json_path, pause_info)

    show_timeline([
        {"step": 1, "artifact": "part_1", "start_sec": 0.0, "end_sec": round(len(p1)/1000, 3), "text": item["part_1"]},
        {"step": 2, "artifact": "pause", "start_sec": round(len(p1)/1000, 3), "end_sec": round(len(p1)/1000 + pause_duration, 3), "text": "[PAUSE]"},
        {"step": 3, "artifact": "part_2", "start_sec": round(len(p1)/1000 + pause_duration, 3), "end_sec": round((len(p1)+len(pause)+len(p2))/1000, 3), "text": item["part_2"]},
    ])
    show_audio("Step 1 - part_1.wav", p1_path)
    display(Markdown(f"**Step 2 - pause**: {pause_duration}s silence, xem mốc trong bảng timeline."))
    show_audio("Step 3 - part_2.wav", p2_path)
    show_audio("Step 4 - input.wav", input_path)
    show_json("Annotation", pause_json_path)

    pause_rows.append({"sample_id": sample_id, "pause_start": pause_info[0]["timestamp"][0], "pause_end": pause_info[0]["timestamp"][1], "input_sec": len(input_sound)/1000})

pause_generation_df = pd.DataFrame(pause_rows)
display(Markdown("### Pause generation summary"))
display(pause_generation_df)


## 6. Step trace: Turn Taking

Mỗi sample hiển thị audio câu user, `input.wav` tự nhiên kết thúc khi user dứt câu, và `turn_taking.json`.

In [ ]:
turn_rows = []
for item in load_template("candor_turn_taking"):
    sample_id = item["id"]
    sample_dir = V1_OUTPUT / "candor_turn_taking" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("candor_turn_taking", sample_id, item.get("description", ""))

    raw_path = quiet_generate(item["text"], sample_dir / "step_1_user_turn.wav", role="primary")
    raw = mixer.load_audio(str(raw_path))
    input_sound = raw
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    turn_info = [{"text": "[TURN-TAKING]", "timestamp": [len(raw) / 1000.0, 0.0]}]
    turn_json_path = sample_dir / "turn_taking.json"
    write_json(turn_json_path, turn_info)

    show_timeline([
        {"step": 1, "artifact": "user_turn", "start_sec": 0.0, "end_sec": round(len(raw)/1000, 3), "text": item["text"]},
    ])
    show_audio("Step 1 - user turn", raw_path)
    show_audio("Step 2 - input.wav", input_path)
    show_json("Annotation", turn_json_path)

    turn_rows.append({"sample_id": sample_id, "turn_end_sec": turn_info[0]["timestamp"][0], "input_sec": len(input_sound)/1000})

turn_generation_df = pd.DataFrame(turn_rows)
display(Markdown("### Turn-taking generation summary"))
display(turn_generation_df)


## 7. Step trace: User Interruption

Mỗi sample hiển thị:

1. `context.wav`
2. `interrupt.wav`
3. vị trí overlay trong cửa sổ agent response
4. `input.wav`
5. `interrupt.json`

In [ ]:
interrupt_rows = []
for item in load_template("synthetic_user_interruption"):
    sample_id = item["id"]
    sample_dir = V1_OUTPUT / "synthetic_user_interruption" / sample_id
    sample_dir.mkdir(parents=True, exist_ok=True)
    sample_header("synthetic_user_interruption", sample_id, item.get("description", ""))

    context_path = quiet_generate(item["context"], sample_dir / "context.wav", role="primary")
    context_profile = generator.last_synthesis.get("profile") if generator.last_synthesis else None
    context_voice = context_profile.get("voice") if context_profile else None
    interrupt_path = quiet_generate(item["interrupt"], sample_dir / "interrupt.wav", voice=context_voice, role="user_interruption")

    context = mixer.load_audio(str(context_path))
    interrupt = mixer.load_audio(str(interrupt_path))
    delay_sec = float(item["interrupt_delay_sec"])
    interrupt_pos_ms = int(delay_sec * 1000)
    silence_window = AudioSegment.silent(duration=interrupt_pos_ms + len(interrupt), frame_rate=16000)
    silence_with_interrupt = silence_window.overlay(interrupt, position=interrupt_pos_ms)
    input_sound = context + silence_with_interrupt
    input_path = save_audio(input_sound, sample_dir / "input.wav")

    start = len(context) / 1000.0 + delay_sec
    end = start + len(interrupt) / 1000.0
    interrupt_info = [{"context": item["context"], "interrupt": item["interrupt"], "timestamp": [start, end]}]
    interrupt_json_path = sample_dir / "interrupt.json"
    write_json(interrupt_json_path, interrupt_info)

    show_timeline([
        {"step": 1, "artifact": "context", "start_sec": 0.0, "end_sec": round(len(context)/1000, 3), "text": item["context"]},
        {"step": 2, "artifact": "pre_interrupt_gap", "start_sec": round(len(context)/1000, 3), "end_sec": round(start, 3), "text": "silence before interrupt"},
        {"step": 3, "artifact": "interrupt_overlay", "start_sec": round(start, 3), "end_sec": round(end, 3), "text": item["interrupt"]},
    ])
    show_audio("Step 1 - context.wav", context_path)
    show_audio("Step 2 - interrupt.wav", interrupt_path)
    show_audio("Step 3 - input.wav", input_path)
    show_json("Annotation", interrupt_json_path)

    interrupt_rows.append({"sample_id": sample_id, "interrupt_start": start, "interrupt_end": end, "input_sec": len(input_sound)/1000})

interrupt_generation_df = pd.DataFrame(interrupt_rows)
display(Markdown("### Interruption generation summary"))
display(interrupt_generation_df)


## 7.1 Test tải PhoWhisper ASR model

Chạy cell này trước cell Gemini/ASR nếu muốn kiểm tra riêng bước tải PhoWhisper. Cell chỉ load model một lần và in trạng thái, chưa transcribe audio.


In [ ]:
import os
import sys
from IPython.display import Markdown, display

RUN_PHOWHISPER_LOAD_TEST = True

if RUN_PHOWHISPER_LOAD_TEST:
    if str(ROOT_DIR / "v1_v1.5" / "get_transcript") not in sys.path:
        sys.path.insert(0, str(ROOT_DIR / "v1_v1.5" / "get_transcript"))
    from asr import ASR_MODEL_ID, get_asr_pipeline

    display(Markdown(f"**Loading PhoWhisper:** `{ASR_MODEL_ID}`"))
    pipe = get_asr_pipeline()
    model_device = getattr(pipe.model, "device", "unknown")
    model_dtype = getattr(pipe.model, "dtype", "unknown")
    display(Markdown(f"**PhoWhisper ready** — device: `{model_device}`, dtype: `{model_dtype}`"))
else:
    display(Markdown("PhoWhisper load test skipped."))


## 8. Gemini inference: tạo `output.wav`, rồi ASR ra `output.json`

Phần mô phỏng v1 ở các cell trên chỉ tạo `input.wav` và annotation. Cell này mới là bước model thật:

- `RUN_GEMINI_INFERENCE = True`: stream từng `input.wav` vào Gemini và ghi `output.wav` như agent-only stem cùng timeline từ 0s, có silence trước lúc Gemini bắt đầu nói.
- `RUN_ASR_FOR_OUTPUT = True`: dùng ASR chuyển `output.wav` thành `output.json` có word timestamp trên timeline chung.
- Cần đặt `GEMINI_API_KEY` trong Kaggle Secrets hoặc biến môi trường trước khi chạy.
- `USE_MOCK_EVAL_OUTPUTS` mặc định `False`; chỉ bật khi muốn debug evaluator mà chưa chạy được Gemini/ASR.


In [ ]:
# [Kaggle Only] Cài đặt các thư viện cần thiết cho ASR Ensemble
!pip install -q chunkformer transformers google-generativeai soundfile python-dotenv


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Audio, Markdown, display

RUN_GEMINI_INFERENCE = bool(os.getenv("GEMINI_API_KEY"))
GEMINI_SCRIPT = ROOT_DIR / "v1_v1.5" / "model_inference" / "gemini" / "inference_gemini25_native.py"
ASR_SCRIPT = ROOT_DIR / "v1_v1.5" / "get_transcript" / "asr.py"
V1_TASKS_FOR_INFERENCE = [
    ("synthetic_pause_handling", "default"),
    ("candor_turn_taking", "default"),
    ("synthetic_user_interruption", "user_interruption"),
]

RUN_ASR_FOR_OUTPUT = RUN_GEMINI_INFERENCE  # Đổi False nếu chỉ muốn nghe output.wav và bỏ qua ASR/evaluation.
USE_MOCK_EVAL_OUTPUTS = False
OVERWRITE_GEMINI_OUTPUT = True
MAX_RESPONSE_SEC = 30

if RUN_GEMINI_INFERENCE:
    for task_name, _asr_task in V1_TASKS_FOR_INFERENCE:
        task_dir = V1_OUTPUT / task_name
        if not task_dir.exists():
            continue
        print(f"Gemini inference -> {task_name}")
        cmd = [
            sys.executable,
            str(GEMINI_SCRIPT),
            "--base-dir", str(V1_OUTPUT),
            "--task", task_name,
            "--max-response-sec", str(MAX_RESPONSE_SEC),
        ]
        if OVERWRITE_GEMINI_OUTPUT:
            cmd.append("--overwrite")
        subprocess.run(cmd, check=True, cwd=str(GEMINI_SCRIPT.parent))
else:
    display(Markdown("**Gemini skipped:** chưa có `GEMINI_API_KEY`. Đầu vào đã được tạo, nhưng chưa có `output.wav` thật."))

def show_gemini_outputs_now(stage_label="Gemini output.wav"):
    rows = []
    for task_name, _asr_task in V1_TASKS_FOR_INFERENCE:
        task_dir = V1_OUTPUT / task_name
        if not task_dir.exists():
            continue
        for sample_dir in sorted(p for p in task_dir.iterdir() if p.is_dir()):
            output_wav = sample_dir / "output.wav"
            combined_wav = sample_dir / "combined.wav"
            if output_wav.exists():
                rows.append({"task": task_name, "sample_id": sample_dir.name, "output_wav": output_wav, "combined_wav": combined_wav if combined_wav.exists() else None})
    if rows:
        display(Markdown(f"### {stage_label} - nghe trực tiếp trước ASR"))
        for row in rows:
            display(Markdown(f"**{row['task']}/{row['sample_id']}/output.wav**"))
            display(Audio(str(row["output_wav"])))
            if row.get("combined_wav"):
                display(Markdown(f"**{row['task']}/{row['sample_id']}/combined.wav** (Full-Duplex)"))
                display(Audio(str(row["combined_wav"])))
    else:
        display(Markdown("Chưa có `output.wav` để nghe. Kiểm tra `GEMINI_API_KEY` hoặc log Gemini inference phía trên."))


show_gemini_outputs_now()

if RUN_ASR_FOR_OUTPUT:
    print("ASR output.wav -> output.json: Ensemble 3 Models (PhoWhisper, Whisper-v3, Chunkformer)")
    asr_module_dir = str(ROOT_DIR / "v1_v1.5" / "get_transcript")
    if asr_module_dir not in sys.path:
        sys.path.insert(0, asr_module_dir)
    import importlib
    import asr_ensemble as asr
    asr = importlib.reload(asr)

    asr.transcribe_v1_benchmark(str(V1_OUTPUT), "output.wav")

if USE_MOCK_EVAL_OUTPUTS:
    for sample_dir in sorted((V1_OUTPUT / "synthetic_pause_handling").iterdir()):
        if sample_dir.is_dir():
            write_json(sample_dir / "output.json", {"text": "", "chunks": []})
    for sample_dir in sorted((V1_OUTPUT / "candor_turn_taking").iterdir()):
        if sample_dir.is_dir():
            start = read_json(sample_dir / "turn_taking.json")[0]["timestamp"][0] + 0.6
            write_json(sample_dir / "output.json", transcript_to_output_json("Tôi đã hiểu câu hỏi và sẽ trả lời ngay.", start=start))
    for sample_dir in sorted((V1_OUTPUT / "synthetic_user_interruption").iterdir()):
        if sample_dir.is_dir():
            start = read_json(sample_dir / "interrupt.json")[0]["timestamp"][1] + 0.7
            write_json(sample_dir / "output.json", transcript_to_output_json("Được rồi, tôi sẽ đổi theo yêu cầu mới của bạn.", start=start))
    print("Mock output.json created for evaluator debugging only.")

rows = []
for task_dir in sorted(V1_OUTPUT.iterdir()):
    if task_dir.is_dir():
        for sample_dir in sorted(p for p in task_dir.iterdir() if p.is_dir()):
            rows.append({
                "task": task_dir.name,
                "sample_id": sample_dir.name,
                "input.wav": (sample_dir / "input.wav").exists(),
                "output.wav": (sample_dir / "output.wav").exists(),
                "output.json": (sample_dir / "output.json").exists(),
                "combined.wav": (sample_dir / "combined.wav").exists(),
                "timing.json": (sample_dir / "inference_timing.json").exists(),
            })
status_df = pd.DataFrame(rows)
display(status_df)

ready_outputs = status_df[status_df["output.wav"]] if len(status_df) else status_df
if len(ready_outputs):
    display(Markdown("### Gemini output.wav - nghe toàn bộ phản hồi"))
    for row in ready_outputs.itertuples(index=False):
        sample_dir = V1_OUTPUT / getattr(row, "task") / getattr(row, "sample_id")
        display(Markdown(f"**{getattr(row, 'task')}/{getattr(row, 'sample_id')}/output.wav**"))
        display(Audio(str(sample_dir / "output.wav")))
        combined_wav = sample_dir / "combined.wav"
        if combined_wav.exists():
            display(Markdown(f"**{getattr(row, 'task')}/{getattr(row, 'sample_id')}/combined.wav** (Full-Duplex)"))
            display(Audio(str(combined_wav)))
        out_json = sample_dir / "output.json"
        if out_json.exists():
            out = read_json(out_json)
            display(Markdown(f"Transcript ASR: `{out.get('text', '')}`"))
else:
    display(Markdown("Chưa có `output.wav` để nghe. Kiểm tra `GEMINI_API_KEY` hoặc lỗi Gemini inference ở log phía trên."))


## 9. Evaluation trace: Pause Handling

Metric chính trong evaluator gốc: `Average take turn`. Với pause handling, `TOR=1` nghĩa là model đã trả lời trong lúc user đang pause; `TOR=0` nghĩa là model không cướp lượt trong pause.

In [ ]:
def chunks_duration(chunks):
    if not chunks:
        return 0.0
    start = chunks[0]["timestamp"][0]
    end = chunks[-1]["timestamp"][-1]
    if end is None:
        end = chunks[-1]["timestamp"][0]
    return max(0.0, end - start)


def tor_from_chunks(chunks, turn_duration_threshold=1, turn_num_words_threshold=3):
    if len(chunks) == 0:
        return 0
    duration = chunks_duration(chunks)
    if duration < turn_duration_threshold and len(chunks) <= turn_num_words_threshold:
        return 0
    return 1

pause_eval_rows = []
missing_pause = []
for sample_dir in sorted((V1_OUTPUT / "synthetic_pause_handling").iterdir()):
    if not sample_dir.is_dir():
        continue
    out_path = sample_dir / "output.json"
    if not out_path.exists():
        missing_pause.append(sample_dir.name)
        continue
    out = read_json(out_path)
    ann = read_json(sample_dir / "pause.json")[0]
    tor = tor_from_chunks(out["chunks"])
    first_start = out["chunks"][0]["timestamp"][0] if out["chunks"] else None
    pause_eval_rows.append({
        "sample_id": sample_dir.name,
        "pause_start": ann["timestamp"][0],
        "pause_end": ann["timestamp"][1],
        "first_output_start": first_start,
        "starts_during_pause": first_start is not None and ann["timestamp"][0] <= first_start <= ann["timestamp"][1],
        "output_text": out.get("text", ""),
        "num_chunks": len(out["chunks"]),
        "TOR": tor,
    })

pause_eval_df = pd.DataFrame(pause_eval_rows)
display(pause_eval_df)
if len(pause_eval_df):
    display(Markdown(f"**Average take turn:** `{pause_eval_df['TOR'].mean():.3f}`"))
else:
    display(Markdown("**Pause evaluation skipped:** chưa có `output.json`. Hãy chạy Gemini + ASR hoặc bật mock fallback."))
if missing_pause:
    display(Markdown(f"Missing output.json: `{len(missing_pause)}` samples"))


## 10. Evaluation trace: Smooth Turn Taking

Metric chính:

- `TOR`: model có trả lời sau khi user kết thúc lượt không.
- `latency`: thời điểm model bắt đầu trả lời trừ thời điểm user kết thúc lượt.

In [ ]:
turn_eval_rows = []
missing_turn = []
for sample_dir in sorted((V1_OUTPUT / "candor_turn_taking").iterdir()):
    if not sample_dir.is_dir():
        continue
    out_path = sample_dir / "output.json"
    if not out_path.exists():
        missing_turn.append(sample_dir.name)
        continue
    out = read_json(out_path)
    ann = read_json(sample_dir / "turn_taking.json")[0]
    user_end = ann["timestamp"][0]
    first_start = out["chunks"][0]["timestamp"][0] if out["chunks"] else None
    latency = None if first_start is None else first_start - user_end
    turn_eval_rows.append({
        "sample_id": sample_dir.name,
        "user_end": user_end,
        "first_output_start": first_start,
        "latency_sec": latency,
        "output_text": out.get("text", ""),
        "num_chunks": len(out["chunks"]),
    })

turn_eval_df = pd.DataFrame(turn_eval_rows)
display(turn_eval_df)
valid_latencies = turn_eval_df["latency_sec"].dropna() if len(turn_eval_df) else pd.Series(dtype=float)
if len(valid_latencies):
    display(Markdown(f"**Average response latency:** `{valid_latencies.mean():.3f}s`"))
else:
    display(Markdown("**Turn-taking evaluation skipped:** chưa có timestamp trong `output.json`."))
if missing_turn:
    display(Markdown(f"Missing output.json: `{len(missing_turn)}` samples"))


## 11. Evaluation trace: User Interruption

Hiện tại benchmark này dùng Gemini để tạo `output.wav`, ASR để tạo `output.json`, và có thể dùng Gemini để chấm semantic rating. Cell này trace phần cấu trúc có thể xem ngay:

- `TOR`: model có trả lời sau interrupt không.
- `latency`: model bắt đầu trả lời sau khi interrupt kết thúc bao lâu.
- `rating`: nếu đã có `rating.json` từ evaluator gốc thì hiển thị; nếu chưa có thì để trống.

In [ ]:
interrupt_eval_rows = []
missing_interrupt = []
for sample_dir in sorted((V1_OUTPUT / "synthetic_user_interruption").iterdir()):
    if not sample_dir.is_dir():
        continue
    out_path = sample_dir / "output.json"
    if not out_path.exists():
        missing_interrupt.append(sample_dir.name)
        continue
    out = read_json(out_path)
    ann = read_json(sample_dir / "interrupt.json")[0]
    interrupt_end = ann["timestamp"][1]
    output_start = out["chunks"][0]["timestamp"][0] if out["chunks"] else None
    latency = None if output_start is None else output_start - interrupt_end
    tor = tor_from_chunks(out["chunks"])
    rating_path = sample_dir / "rating.json"
    rating = read_json(rating_path)["rating"] if rating_path.exists() else None
    interrupt_eval_rows.append({
        "sample_id": sample_dir.name,
        "context": ann["context"],
        "interrupt": ann["interrupt"],
        "interrupt_end": interrupt_end,
        "output_start": output_start,
        "output_text": out.get("text", ""),
        "TOR": tor,
        "latency_sec": latency,
        "rating_if_available": rating,
    })

interrupt_eval_df = pd.DataFrame(interrupt_eval_rows)
display(interrupt_eval_df)
if len(interrupt_eval_df):
    display(Markdown(f"**Average take turn:** `{interrupt_eval_df['TOR'].mean():.3f}`"))
valid_latencies = interrupt_eval_df["latency_sec"].dropna() if len(interrupt_eval_df) else pd.Series(dtype=float)
if len(valid_latencies):
    display(Markdown(f"**Average latency:** `{valid_latencies.mean():.3f}s`"))
if len(interrupt_eval_df) and interrupt_eval_df["rating_if_available"].notna().any():
    display(Markdown(f"**Average rating:** `{interrupt_eval_df['rating_if_available'].dropna().mean():.3f}`"))
else:
    display(Markdown("**Average rating:** chưa có `rating.json`; bật evaluator gốc ở cuối notebook để chấm semantic rating bằng Gemini."))
if missing_interrupt:
    display(Markdown(f"Missing output.json: `{len(missing_interrupt)}` samples"))


## 12. Optional: chạy evaluator gốc

Các cell trên đã trace công thức và kết quả per-sample bằng `output.wav/output.json` từ Gemini. Nếu muốn đối chiếu với evaluator gốc của repo, bật `RUN_ORIGINAL_EVALUATORS=True`. Task `user_interruption` hiện dùng `GEMINI_API_KEY` để chấm semantic rating.

In [ ]:
import subprocess

RUN_ORIGINAL_EVALUATORS = False
EVAL_SCRIPT = ROOT_DIR / "v1_v1.5" / "evaluation" / "evaluate.py"

if RUN_ORIGINAL_EVALUATORS:
    commands = [
        [sys.executable, str(EVAL_SCRIPT), "--task", "pause_handling", "--root_dir", str(V1_OUTPUT / "synthetic_pause_handling")],
        [sys.executable, str(EVAL_SCRIPT), "--task", "smooth_turn_taking", "--root_dir", str(V1_OUTPUT / "candor_turn_taking")],
    ]
    if os.getenv("GEMINI_API_KEY"):
        commands.append([sys.executable, str(EVAL_SCRIPT), "--task", "user_interruption", "--root_dir", str(V1_OUTPUT / "synthetic_user_interruption")])
    else:
        print("Skip original user_interruption evaluator: missing GEMINI_API_KEY")

    for cmd in commands:
        print("$", " ".join(cmd))
        result = subprocess.run(cmd, text=True, capture_output=True)
        print(result.stdout[-1200:])
        if result.stderr:
            print(result.stderr[-1200:])
else:
    print("RUN_ORIGINAL_EVALUATORS=False; skipped.")
